# Workflow 4 — kNN-CDF vs 2PCF sensitivity comparison

Two parts:
1. Generate ξ(r) for the same selections already covered by the kNN
   pipeline (workflows 1-2)
2. Head-to-head comparison of kNN-CDF vs 2PCF sensitivity, with both
   summary statistics standardised (z-scored per bin) so R(p) is in
   units of cross-simulation sigma and directly comparable.

Uses `load_and_analyze(..., standardize_flag=True)` from
`src/analysis_pipeline.py` instead of a local re-implementation.

## Part 1 — Generate ξ(r)

Mirrors the selections already generated for kNN in workflows 1-2.
Set `GENERATE_2PCF = False` to skip if the .npz files already exist.

In [1]:
import sys
sys.path.append("..")

from src.generate_2pcf import run_suite

GENERATE_2PCF = False

if GENERATE_2PCF:

    # 1. Mass-selected at z~1 (snap 50)
    for mcut in [1e6, 1e7, 1e8]:
        print(f"\n{'='*50}")
        print(f"  mass > {mcut:.0e}  snap=50")
        print(f"{'='*50}")
        run_suite(snap=50, selection="mass", mass_cut=mcut)

    # 2. Redshift evolution (mass > 1e6)
    for snap in [32, 90]:
        print(f"\n{'='*50}")
        print(f"  mass > 1e6  snap={snap}")
        print(f"{'='*50}")
        run_suite(snap=snap, selection="mass", mass_cut=1e6)

    # 3. Activity-selected at z~1.
    # NOTE: matches workflow 2 -- only the 10% luminosity threshold is
    # generated (not 2%), since generate_knn_active.py's output filename
    # doesn't encode top_fraction and a second threshold would silently
    # overwrite this one. Keep this in sync with workflow 2's SELECTIONS.
    print(f"\n{'='*50}")
    print(f"  luminosity top-10%  snap=50")
    print(f"{'='*50}")
    run_suite(snap=50, selection="luminosity", mass_cut=1e6, top_fraction=0.10)

    print(f"\n{'='*50}")
    print(f"  fEdd top-10%  snap=50")
    print(f"{'='*50}")
    run_suite(snap=50, selection="fedd", mass_cut=1e6, top_fraction=0.10)

    print("\n\nAll 2PCF generation done.")

## Part 2 — kNN vs 2PCF comparison

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.analysis_pipeline import load_and_analyze, standardize
from src.remove_abundance import remove_abundance
from src.parameter_sensitivity import load_params
from src.sensitivity_bootstrap import sensitivity_table, _rms_response

### Config

In [3]:
PARAM_FILE = (
    "../CAMELS-master/docs/params/IllustrisTNG/"
    "CosmoAstroSeed_IllustrisTNG_L25n256_LH.txt"
)
OUTPUT_DIR = "../outputs"
FIG_DIR    = "../plots"

PARAMS = [
    "Omega_m", "sigma_8",
    "A_SN1",   "A_AGN1",
    "A_SN2",   "A_AGN2",
]

# Quick mode for iterating on the notebook; this comparison is the most
# expensive notebook in the pipeline (kNN AND 2PCF tables x 5 selections),
# so QUICK is especially worth using while developing.
QUICK = False
N_BOOT, N_NULL = (200, 200) if QUICK else (5000, 5000)
print(f"QUICK={QUICK}  N_BOOT={N_BOOT}  N_NULL={N_NULL}")

# "L10" (not "L2") to match workflow 2/3: only the top-10% luminosity
# threshold is generated. If you ever add a top-2% run, give it its own
# output filename (generate_knn_active.py doesn't encode top_fraction)
# and a new key here, e.g. "L2".
SELECTIONS = {
    "M1e6": (
        f"{OUTPUT_DIR}/knn_snap50_M1e+06.npz",
        f"{OUTPUT_DIR}/2pcf_snap50_M1e+06.npz",
    ),
    "M1e7": (
        f"{OUTPUT_DIR}/knn_snap50_M1e+07.npz",
        f"{OUTPUT_DIR}/2pcf_snap50_M1e+07.npz",
    ),
    "M1e8": (
        f"{OUTPUT_DIR}/knn_snap50_M1e+08.npz",
        f"{OUTPUT_DIR}/2pcf_snap50_M1e+08.npz",
    ),
    "fEdd10": (
        f"{OUTPUT_DIR}/knn_fedd_snap50_M1e+06.npz",
        f"{OUTPUT_DIR}/2pcf_fedd_snap50_M1e+06.npz",
    ),
    "L10": (
        f"{OUTPUT_DIR}/knn_luminosity_snap50_M1e+06.npz",
        f"{OUTPUT_DIR}/2pcf_luminosity_snap50_M1e+06.npz",
    ),
}

QUICK=False  N_BOOT=5000  N_NULL=5000


### 1. Compute sensitivity tables (standardised)

In [4]:
knn_tables  = {}
tpcf_tables = {}

for sel, (knn_path, tpcf_path) in SELECTIONS.items():

    print(f"\n{'='*60}")
    print(f"  {sel}")
    print(f"{'='*60}")

    for label, path, store in [
        ("kNN",  knn_path,  knn_tables),
        ("2PCF", tpcf_path, tpcf_tables),
    ]:
        try:
            print(f"  {label:5s} ... ", end="", flush=True)
            result = load_and_analyze(
                path, params=PARAMS, params_file=PARAM_FILE,
                standardize_flag=True,
            )
            df = sensitivity_table(
                result["residuals"], result["theta"],
                params=PARAMS,
                n_boot=N_BOOT,
                n_null=N_NULL,
            )
            store[sel] = df
            print("done")
        except FileNotFoundError:
            print(f"[skip] {path}")


  M1e6
  kNN   ... 

/home/jovyan/home/notebooks/../src/parameter_sensitivity.py:13: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  params = pd.read_csv(


done
  2PCF  ... 

/home/jovyan/home/notebooks/../src/parameter_sensitivity.py:13: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  params = pd.read_csv(


done

  M1e7
  kNN   ... 

...

done

  L10
  kNN   ... 

/home/jovyan/home/notebooks/../src/parameter_sensitivity.py:13: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  params = pd.read_csv(


done
  2PCF  ... 

/home/jovyan/home/notebooks/../src/parameter_sensitivity.py:13: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  params = pd.read_csv(


done


### 2. Side-by-side bar chart (paper figure)

In [5]:
common = sorted(set(knn_tables) & set(tpcf_tables))

if common:

    fig, axes = plt.subplots(
        len(common), 1,
        figsize=(8, 3.0 * len(common)),
        sharex=False,
    )
    if len(common) == 1:
        axes = [axes]

    def safe_err(df, params):
        lo = [max(df.loc[p, "R_obs"] - df.loc[p, "ci_lo"], 0) for p in params]
        hi = [max(df.loc[p, "ci_hi"] - df.loc[p, "R_obs"], 0) for p in params]
        return lo, hi

    for ax, sel in zip(axes, common):

        knn_df  = knn_tables[sel].set_index("parameter")
        tpcf_df = tpcf_tables[sel].set_index("parameter")

        y      = np.arange(len(PARAMS))
        height = 0.35

        knn_vals = [knn_df.loc[p, "R_obs"] for p in PARAMS]
        knn_lo, knn_hi = safe_err(knn_df, PARAMS)

        tpcf_vals = [tpcf_df.loc[p, "R_obs"] for p in PARAMS]
        tpcf_lo, tpcf_hi = safe_err(tpcf_df, PARAMS)

        ax.barh(
            y + height/2, knn_vals, height=height,
            xerr=[knn_lo, knn_hi],
            color="#2c3e50", edgecolor="white", linewidth=0.4,
            capsize=2, error_kw=dict(lw=1),
            label="kNN-CDF", zorder=3,
        )
        ax.barh(
            y - height/2, tpcf_vals, height=height,
            xerr=[tpcf_lo, tpcf_hi],
            color="#e67e22", edgecolor="white", linewidth=0.4,
            capsize=2, error_kw=dict(lw=1),
            label=r"$\xi(r)$", zorder=3,
        )

        null_knn  = knn_df["null_floor"].median()
        null_tpcf = tpcf_df["null_floor"].median()
        ax.axvline(null_knn,  color="#2c3e50", ls=":", lw=1, alpha=0.4)
        ax.axvline(null_tpcf, color="#e67e22", ls=":", lw=1, alpha=0.4)

        ax.set_yticks(y)
        ax.set_yticklabels(PARAMS, fontsize=9)
        ax.set_title(sel, fontweight="bold", fontsize=11)
        ax.legend(fontsize=8, loc="lower right")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    axes[-1].set_xlabel(r"Standardised RMS response $\tilde{R}_p$")
    fig.tight_layout()

    fig.savefig(f"{FIG_DIR}/knn_vs_2pcf_standardised.pdf", dpi=300)
    fig.savefig(f"{FIG_DIR}/knn_vs_2pcf_standardised.png", dpi=150)
    plt.close(fig)
    print(f"\nSaved: {FIG_DIR}/knn_vs_2pcf_standardised.pdf")
else:
    print("[skip] no selections with both kNN and 2PCF tables available")


Saved: ../plots/knn_vs_2pcf_standardised.pdf


### 3. Ratio table

In [6]:
def rms_ratio(rk, rt):
    """R_kNN / R_2PCF, with a guard against near-zero denominators."""
    return rk / rt if rt > 1e-10 else np.inf


rows = []
for sel in common:
    knn_df  = knn_tables[sel].set_index("parameter")
    tpcf_df = tpcf_tables[sel].set_index("parameter")

    for p in PARAMS:
        rk = knn_df.loc[p, "R_obs"]
        rt = tpcf_df.loc[p, "R_obs"]

        rows.append({
            "selection": sel,
            "parameter": p,
            "R_kNN":     rk,
            "R_2PCF":    rt,
            "ratio":     rms_ratio(rk, rt),
            "sig_kNN":   knn_df.loc[p, "significant"],
            "sig_2PCF":  tpcf_df.loc[p, "significant"],
            "kNN_only":  knn_df.loc[p, "significant"] and not tpcf_df.loc[p, "significant"],
            "2PCF_only": tpcf_df.loc[p, "significant"] and not knn_df.loc[p, "significant"],
        })

ratio_df = pd.DataFrame(rows)

print("\n" + "=" * 70)
print("  Standardised R(p) — kNN vs 2PCF")
print("=" * 70)

pivot = ratio_df.pivot_table(
    index="parameter",
    columns="selection",
    values=["R_kNN", "R_2PCF"],
)
print(pivot.to_string(float_format="%.4f"))

print("\n\nRatio  R_kNN / R_2PCF  (>1 means kNN wins):")
print("-" * 55)
ratio_pivot = ratio_df.pivot_table(
    index="parameter",
    columns="selection",
    values="ratio",
)
print(ratio_pivot.to_string(float_format="%.2f"))

knn_only = ratio_df[ratio_df["kNN_only"]]
if len(knn_only) > 0:
    print("\n\nDetected by kNN but NOT by 2PCF:")
    print("-" * 55)
    for _, row in knn_only.iterrows():
        print(
            f"  {row['selection']:8s}  {row['parameter']:10s}"
            f"  R_kNN={row['R_kNN']:.4f}  R_2PCF={row['R_2PCF']:.4f}"
            f"  ratio={row['ratio']:.2f}x"
        )

tpcf_only = ratio_df[ratio_df["2PCF_only"]]
if len(tpcf_only) > 0:
    print("\n\nDetected by 2PCF but NOT by kNN:")
    print("-" * 55)
    for _, row in tpcf_only.iterrows():
        print(
            f"  {row['selection']:8s}  {row['parameter']:10s}"
            f"  R_kNN={row['R_kNN']:.4f}  R_2PCF={row['R_2PCF']:.4f}"
            f"  ratio={row['ratio']:.2f}x"
        )

if len(knn_only) == 0 and len(tpcf_only) == 0:
    print("\n  No exclusive detections — both statistics detect the same parameters.")


  Standardised R(p) — kNN vs 2PCF
          R_2PCF                              R_kNN                            
selection    L10   M1e6   M1e7   M1e8 fEdd10    L10   M1e6   M1e7   M1e8 fEdd10
parameter                                                                      
A_AGN1    0.0908 0.0900 0.1078 0.1214 0.1056 0.1375 0.1235 0.0579 0.0875 0.1169
A_AGN2    0.1079 0.0755 0.1171 0.1335 0.1062 0.0659 0.0533 0.1452 0.0679 0.0725
A_SN1     0.0893 0.4661 0.1333 0.1327 0.1222 0.1469 0.3296 0.2182 0.0914 0.1825
A_SN2     0.1269 0.1948 0.1086 0.1095 0.1194 0.1992 0.1606 0.1387 0.0763 0.2646
Omega_m   0.0460 0.1396 0.2785 0.2505 0.0648 0.3532 0.4021 0.2632 0.2759 0.3558
sigma_8   0.0868 0.1390 0.1368 0.1136 0.0803 0.0940 0.2101 0.3178 0.3002 0.0941


Ratio  R_kNN / R_2PCF  (>1 means kNN wins):
-------------------------------------------------------
selection  L10  M1e6  M1e7  M1e8  fEdd10
parameter                               
A_AGN1    1.52  1.37  0.54  0.72    1.11
A_AGN2    0.61  0.71

In [7]:
from src.selection_bias import (
    compute_ipw_weights, weighted_sensitivity_table,
    residualize_theta_on_nbh,
)
from src.parameter_sensitivity import load_params

all_ids = np.arange(1000)
theta_all = load_params(all_ids, PARAM_FILE)

# Sanity check: the TNG luminosity file was overwritten by a top-2% run at
# some point (generate_knn_active doesn't encode top_fraction in the
# filename). Top-10% should be ~1000 sims / mean nbh ~115; top-2% shows
# ~999 / ~23. Bail out rather than silently analysing the wrong selection.
_lum = np.load(SELECTIONS["L10"][0], allow_pickle=True)
print(f"L10 file: {len(_lum['sim_ids'])} sims, mean nbh = {_lum['nbh'].mean():.2f}")
assert _lum["nbh"].mean() > 60, (
    "TNG L10 file looks like a top-2% run, not top-10%. Regenerate before using."
)

corrected_ratio_rows = []

for sel in ["L10", "fEdd10"]:
    if sel not in common:
        continue

    knn_path, tpcf_path = SELECTIONS[sel]
    corrected = {}

    for stat, path in [("kNN", knn_path), ("2PCF", tpcf_path)]:
        res = load_and_analyze(
            path, params=PARAMS, params_file=PARAM_FILE,
            standardize_flag=True,
        )
        # TNG has ~zero whole-sim dropout, so this short-circuits to uniform
        # weights and the call reduces to the residualization-only correction
        # already validated by the TNG permutation control.
        weights, _, _ = compute_ipw_weights(
            retained_ids=res["sim_ids"], all_ids=all_ids,
            theta_all=theta_all, params=PARAMS,
        )
        theta_resid = residualize_theta_on_nbh(res["theta"], res["nbh"], params=PARAMS)
        corrected[stat] = weighted_sensitivity_table(
            res["residuals"], theta_resid, weights,
            params=PARAMS, n_boot=N_BOOT, n_null=N_NULL,
        ).set_index("parameter")

    for p in PARAMS:
        rk = corrected["kNN"].loc[p, "R_obs"]
        rt = corrected["2PCF"].loc[p, "R_obs"]
        nk = corrected["kNN"].loc[p, "null_floor"]
        nt = corrected["2PCF"].loc[p, "null_floor"]
        raw_k = knn_tables[sel].set_index("parameter").loc[p, "R_obs"]
        raw_t = tpcf_tables[sel].set_index("parameter").loc[p, "R_obs"]

        corrected_ratio_rows.append({
            "selection":  sel,
            "parameter":  p,
            "ratio_raw":  rms_ratio(raw_k, raw_t),
            "ratio_corr": rms_ratio(rk, rt),
            "sig_kNN":    corrected["kNN"].loc[p, "significant"],
            "sig_2PCF":   corrected["2PCF"].loc[p, "significant"],
            "snr_kNN":    rk / nk,
            "snr_2PCF":   rt / nt,
            "snr_ratio":  (rk / nk) / (rt / nt) if nt > 0 else np.inf,
        })

corr_ratio_df_tng = pd.DataFrame(corrected_ratio_rows)
print(corr_ratio_df_tng.to_string(index=False, float_format="%.3f"))

L10 file: 1000 sims, mean nbh = 114.88


/home/jovyan/home/notebooks/../src/parameter_sensitivity.py:13: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  params = pd.read_csv(
/home/jovyan/home/notebooks/../src/parameter_sensitivity.py:13: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  params = pd.read_csv(
/home/jovyan/home/notebooks/../src/parameter_sensitivity.py:13: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  params = pd.read_csv(
/home/jovyan/home/notebooks/../src/parameter_sensitivity.py:13: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  params = pd.read_csv(
/home/jovyan/home/notebooks/../src/parameter_sensitivity.py:13: FutureWarning: T

selection parameter  ratio_raw  ratio_corr  sig_kNN  sig_2PCF  snr_kNN  snr_2PCF  snr_ratio
      L10   Omega_m      7.679       2.553     True      True    2.339     1.026      2.280
      L10   sigma_8      1.083       1.862     True     False    1.230     0.742      1.659
      L10     A_SN1      1.645       1.299    False     False    0.975     0.833      1.171
      L10    A_AGN1      1.515       1.424     True     False    1.082     0.846      1.280
      L10     A_SN2      1.570       1.542     True      True    1.609     1.169      1.376
      L10    A_AGN2      0.611       0.667    False      True    0.601     1.013      0.593
   fEdd10   Omega_m      5.490       2.141     True      True    2.385     1.244      1.918
   fEdd10   sigma_8      1.172       1.827     True     False    1.321     0.812      1.626
   fEdd10     A_SN1      1.493       1.267     True      True    1.286     1.142      1.126
   fEdd10    A_AGN1      1.107       1.035    False     False    0.915     0.990

### 4. Matched-sim comparison on standardised residuals

**Bug fix**: the previous version of this cell matched kNN and 2PCF
sims using `np.isin(sim_ids, common_ids)` to mask each array. That
preserves each array's *original* row order, not `common_ids`' order
-- and the two .npz files don't necessarily store sims in the same
order as each other. The result was that `knn_res[i]` and `tpcf_res[i]`
were frequently different simulations, which silently washed out
the real signal (everything came out close to a 1.0x "tie", which is
what you'd expect from randomly mismatched pairs). This cell now
builds an explicit per-array index so every row is aligned to the
same `common_ids` order, with an assertion guarding against regression.

**Note**: unlike sections 1-3, this still uses `_rms_response` directly
(a single point estimate), not the bootstrap CI / null-test machinery
from `sensitivity_table`. There's no uncertainty attached to the
"winner" calls below -- treat this as a cross-check on the bootstrap
result above, not an independently rigorous comparison.

In [8]:
print("\n\n" + "=" * 70)
print("  Matched-sim comparison (standardised, point estimate only)")
print("=" * 70)

for sel in common:
    knn_path, tpcf_path = SELECTIONS[sel]

    try:
        knn_data  = np.load(knn_path,  allow_pickle=True)
        tpcf_data = np.load(tpcf_path, allow_pickle=True)
    except FileNotFoundError:
        continue

    common_ids = np.intersect1d(knn_data["sim_ids"], tpcf_data["sim_ids"])

    # IMPORTANT: np.isin(sim_ids, common_ids) only filters by membership --
    # it does NOT reorder to match common_ids, and knn_data/tpcf_data can
    # each store their sims in a different order. Masking both arrays with
    # np.isin and then zipping them row-by-row silently pairs up the WRONG
    # simulations between kNN and 2PCF (verified: this was happening in the
    # previous version of this cell). Build an explicit per-array position
    # lookup instead, so every array is reordered into the same, shared
    # common_ids order.
    knn_pos  = {sid: i for i, sid in enumerate(knn_data["sim_ids"])}
    tpcf_pos = {sid: i for i, sid in enumerate(tpcf_data["sim_ids"])}
    knn_idx  = np.array([knn_pos[sid]  for sid in common_ids])
    tpcf_idx = np.array([tpcf_pos[sid] for sid in common_ids])

    # Sanity check so this can't silently regress again.
    assert np.array_equal(knn_data["sim_ids"][knn_idx], common_ids)
    assert np.array_equal(tpcf_data["sim_ids"][tpcf_idx], common_ids)

    print(f"\n  {sel}:  common={len(common_ids)} sims")

    if len(common_ids) < 50:
        print("    [too few — skip]")
        continue

    knn_res  = standardize(remove_abundance(
        knn_data["summaries"][knn_idx], knn_data["nbh"][knn_idx],
    ))
    tpcf_res = standardize(remove_abundance(
        tpcf_data["summaries"][tpcf_idx], tpcf_data["nbh"][tpcf_idx],
    ))
    theta = load_params(common_ids, PARAM_FILE)

    print(f"    {'param':10s}  {'R_kNN':>8s}  {'R_2PCF':>8s}  {'ratio':>7s}  winner")
    print(f"    {'-'*48}")

    for p in PARAMS:
        rk = _rms_response(knn_res,  theta[p].values)
        rt = _rms_response(tpcf_res, theta[p].values)
        rat = rms_ratio(rk, rt)
        winner = "kNN" if rat > 1 else "2PCF" if rat < 1 else "tie"
        print(f"    {p:10s}  {rk:8.4f}  {rt:8.4f}  {rat:7.2f}x  {winner}")



  Matched-sim comparison (standardised, point estimate only)

  L10:  common=1000 sims


/home/jovyan/home/notebooks/../src/parameter_sensitivity.py:13: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  params = pd.read_csv(
/home/jovyan/home/notebooks/../src/parameter_sensitivity.py:13: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  params = pd.read_csv(


    param          R_kNN    R_2PCF    ratio  winner
    ------------------------------------------------
    Omega_m       0.3532    0.0460     7.68x  kNN
    sigma_8       0.0940    0.0868     1.08x  kNN
    A_SN1         0.1469    0.0893     1.64x  kNN
    A_AGN1        0.1375    0.0908     1.52x  kNN
    A_SN2         0.1992    0.1269     1.57x  kNN
    A_AGN2        0.0659    0.1079     0.61x  2PCF

  M1e6:  common=1000 sims
    param          R_kNN    R_2PCF    ratio  winner
    ------------------------------------------------
    Omega_m       0.4021    0.1396     2.88x  kNN
    sigma_8       0.2101    0.1390     1.51x  kNN
    A_SN1         0.3296    0.4661     0.71x  2PCF
    A_AGN1        0.1235    0.0900     1.37x  kNN
    A_SN2         0.1606    0.1948     0.82x  2PCF
    A_AGN2        0.0533    0.0755     0.71x  2PCF

  M1e7:  common=856 sims


/home/jovyan/home/notebooks/../src/parameter_sensitivity.py:13: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  params = pd.read_csv(
/home/jovyan/home/notebooks/../src/parameter_sensitivity.py:13: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  params = pd.read_csv(


    param          R_kNN    R_2PCF    ratio  winner
    ------------------------------------------------
    Omega_m       0.3212    0.2785     1.15x  kNN
    sigma_8       0.2505    0.1368     1.83x  kNN
    A_SN1         0.1863    0.1333     1.40x  kNN
    A_AGN1        0.0647    0.1078     0.60x  2PCF
    A_SN2         0.1614    0.1086     1.49x  kNN
    A_AGN2        0.1007    0.1171     0.86x  2PCF

  M1e8:  common=599 sims
    param          R_kNN    R_2PCF    ratio  winner
    ------------------------------------------------
    Omega_m       0.3229    0.2505     1.29x  kNN
    sigma_8       0.1592    0.1136     1.40x  kNN
    A_SN1         0.1506    0.1327     1.13x  kNN
    A_AGN1        0.1220    0.1214     1.01x  kNN
    A_SN2         0.0834    0.1095     0.76x  2PCF
    A_AGN2        0.0951    0.1335     0.71x  2PCF

  fEdd10:  common=1000 sims
    param          R_kNN    R_2PCF    ratio  winner
    ------------------------------------------------
    Omega_m       0.3558  

/home/jovyan/home/notebooks/../src/parameter_sensitivity.py:13: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  params = pd.read_csv(


### 5. LaTeX table

In [9]:
if common:
    sel = "L10" if "L10" in common else common[0]
    knn_df  = knn_tables[sel].set_index("parameter")
    tpcf_df = tpcf_tables[sel].set_index("parameter")

    print(f"\n\n% LaTeX comparison table ({sel}, standardised)")
    print(r"\begin{table}")
    print(r"  \centering")
    print(r"  \caption{Standardised RMS sensitivity $\tilde{R}_p$ for the")
    print(f"           {sel} selection at $z \\approx 1$.  Both kNN-CDF")
    print(r"           and $\xi(r)$ residuals are z-scored per radial bin")
    print(r"           before computing $\tilde{R}_p$, making the values")
    print(r"           directly comparable.}")
    print(r"  \label{tab:knn_vs_2pcf}")
    print(r"  \begin{tabular}{lcccc}")
    print(r"    \hline\hline")
    print(r"    Parameter & $\tilde{R}_p^{\rm kNN}$"
          r" & $\tilde{R}_p^{\xi}$"
          r" & Ratio & Exclusive \\")
    print(r"    \hline")

    for p in PARAMS:
        rk = knn_df.loc[p, "R_obs"]
        rt = tpcf_df.loc[p, "R_obs"]
        rat = rms_ratio(rk, rt)
        sk = knn_df.loc[p, "significant"]
        st = tpcf_df.loc[p, "significant"]

        if sk and not st:
            exc = "kNN"
        elif st and not sk:
            exc = r"$\xi$"
        else:
            exc = "---"

        print(
            f"    {p:10s} & "
            f"${rk:.4f}$ & "
            f"${rt:.4f}$ & "
            f"${rat:.1f}\\times$ & "
            f"{exc} \\\\"
        )

    print(r"    \hline")
    print(r"  \end{tabular}")
    print(r"\end{table}")
else:
    print("[skip] no common selections available")

print("\n\nDone.")



% LaTeX comparison table (L10, standardised)
\begin{table}
  \centering
  \caption{Standardised RMS sensitivity $\tilde{R}_p$ for the
           L10 selection at $z \approx 1$.  Both kNN-CDF
           and $\xi(r)$ residuals are z-scored per radial bin
           before computing $\tilde{R}_p$, making the values
           directly comparable.}
  \label{tab:knn_vs_2pcf}
  \begin{tabular}{lcccc}
    \hline\hline
    Parameter & $\tilde{R}_p^{\rm kNN}$ & $\tilde{R}_p^{\xi}$ & Ratio & Exclusive \\
    \hline
    Omega_m    & $0.3532$ & $0.0460$ & $7.7\times$ & kNN \\
    sigma_8    & $0.0940$ & $0.0868$ & $1.1\times$ & --- \\
    A_SN1      & $0.1469$ & $0.0893$ & $1.6\times$ & kNN \\
    A_AGN1     & $0.1375$ & $0.0908$ & $1.5\times$ & kNN \\
    A_SN2      & $0.1992$ & $0.1269$ & $1.6\times$ & --- \\
    A_AGN2     & $0.0659$ & $0.1079$ & $0.6\times$ & --- \\
    \hline
  \end{tabular}
\end{table}


Done.
